In [6]:
import os
import pandas as pd
import numpy as np
import pickle
import yaml

import warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from pydlm import dlm, dynamic


In [7]:
warnings.filterwarnings("ignore")

In [8]:
df = pd.read_csv("../../data/oh.csv")
df["Ngày"] = pd.to_datetime(df["Ngày"], format="%Y-%m-%d")

In [9]:
df.head()

,Tên_mặt_hàng,Ngày,Giá,Thị_trường_An Giang,Thị_trường_Bạc Liêu,Thị_trường_Bến Tre,Thị_trường_Cà Mau,Thị_trường_Cần Thơ,Thị_trường_Gia Lai,Thị_trường_Hà Nội,...,Nguồn_Cờ Đỏ,Nguồn_Giồng Riềng,Nguồn_Long Xuyên,Nguồn_Thành phố Thái Bình,Nguồn_Tri Tôn,Nguồn_Tỉnh An Giang,Nguồn_huyện Kiến Xương,Nguồn_huyện Quỳnh Phụ,Nguồn_huyện Thái Thụy,Nguồn_huyện Đông Hưng
0,Cà phê Robusta nhân xô,2025-05-09,128233.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,Cà phê Robusta nhân xô,2025-05-09,128350.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,Cà phê Robusta nhân xô,2025-05-09,128233.0,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
3,Cà phê Robusta nhân xô,2025-05-09,128200.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,Cà phê Robusta nhân xô,2025-05-09,128000.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [10]:
exog_cols = df.select_dtypes(include="bool").columns
print(len(exog_cols))
exog_cols

48


Index(['Thị_trường_An Giang', 'Thị_trường_Bạc Liêu', 'Thị_trường_Bến Tre',
       'Thị_trường_Cà Mau', 'Thị_trường_Cần Thơ', 'Thị_trường_Gia Lai',
       'Thị_trường_Hà Nội', 'Thị_trường_Hậu Giang', 'Thị_trường_Hồ Chí Minh',
       'Thị_trường_Kiên Giang', 'Thị_trường_Kon Tum', 'Thị_trường_Long An',
       'Thị_trường_Lâm Đồng', 'Thị_trường_Sóc Trăng', 'Thị_trường_Sơn La',
       'Thị_trường_Thái Bình', 'Thị_trường_Tiền Giang', 'Thị_trường_Trà Vinh',
       'Thị_trường_Vĩnh Long', 'Thị_trường_Đắk Lắk', 'Thị_trường_Đắk Nông',
       'Thị_trường_Đồng Tháp', 'Loại_giá_Bán buôn', 'Loại_giá_Bán lẻ',
       'Loại_giá_Bán ra', 'Loại_giá_Công ty thu mua', 'Loại_giá_Khác',
       'Loại_giá_Thu mua', 'Loại_giá_Thu mua tại vườn',
       'Loại_giá_Thương lái thu mua', 'Loại_giá_Tại chợ',
       'Loại_giá_Vựa thu mua', 'Loại_giá_Xuất khẩu', 'Loại_giá_Đại lý thu mua',
       'Nguồn_Bán lẻ', 'Nguồn_CTV Agroinfo/ Tintaynguyen',
       'Nguồn_CTV địa phương', 'Nguồn_Cao Lãnh', 'Nguồn_Cờ Đỏ',
       'Ng

In [6]:
df[exog_cols] = df[exog_cols].astype(int)

In [7]:
df.head()

,Tên_mặt_hàng,Ngày,Giá,Thị_trường_An Giang,Thị_trường_Bạc Liêu,Thị_trường_Bến Tre,Thị_trường_Cà Mau,Thị_trường_Cần Thơ,Thị_trường_Gia Lai,Thị_trường_Hà Nội,...,Nguồn_Cờ Đỏ,Nguồn_Giồng Riềng,Nguồn_Long Xuyên,Nguồn_Thành phố Thái Bình,Nguồn_Tri Tôn,Nguồn_Tỉnh An Giang,Nguồn_huyện Kiến Xương,Nguồn_huyện Quỳnh Phụ,Nguồn_huyện Thái Thụy,Nguồn_huyện Đông Hưng
0,Cà phê Robusta nhân xô,2025-05-09,128233.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Cà phê Robusta nhân xô,2025-05-09,128350.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Cà phê Robusta nhân xô,2025-05-09,128233.0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,Cà phê Robusta nhân xô,2025-05-09,128200.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Cà phê Robusta nhân xô,2025-05-09,128000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
items = df["Tên_mặt_hàng"].unique()

# One-hot

# DLM

In [10]:
for idx, item in enumerate(items):
    item_df = df[df["Tên_mặt_hàng"] == item].sort_values("Ngày")

    y = item_df["Giá"]
    exog_values = item_df[exog_cols]
    exog_values = np.array(exog_values)
    exog_component = dynamic(features=exog_values, discount=0.99, name='exog', w=1.0)

    model = dlm(y) + exog_component
    model.fit()

    with open(f"../../models/dlm_OH/{idx}.pkl", "wb") as file:
        pickle.dump(model, file)

INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization finished.
INFO:pydlm:Starting forward filtering...
INFO:pydlm:Forward filtering completed.
INFO:pydlm:Starting backward smoothing...
INFO:pydlm:Backward smoothing completed.
INFO:pydlm:Initializing models...
INFO:pydlm:Initialization fini

In [11]:
for idx, item in enumerate(items):
    item_df = df[df["Tên_mặt_hàng"] == item].sort_values("Ngày")

    y = item_df["Giá"]
    exog_values = item_df[exog_cols]

    # SARIMAX model training
    model = SARIMAX(
        y,
        exog=exog_values,
        order=(1, 1, 1),           
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    # Save model
    with open(f"../../models/sarimax_OH/{idx}.pkl", "wb") as file:
        pickle.dump(model, file)